# 📊 Dashboard de Inteligencia de Negocio | Future Space
---
**Modulo de Analitica Avanzada v2.0**  
Este informe es autogestionado y no depende de archivos externos para garantizar su ejecucion.

In [16]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
from sqlalchemy import create_engine

# --- CLASE MOTOR DE DATOS INTEGRADA (PARA EVITAR ERRORES DE IMPORTACION) ---
class DataEngine:
    def __init__(self):
        self.db_url = "mysql+mysqlconnector://root:@localhost:3306/practica"
        try:
            self.engine = create_engine(self.db_url)
            with self.engine.connect() as conn: pass
            self.is_mock = False
            print("Conexion a DB real exitosa.")
        except:
            self.is_mock = True
            print("Modo DEMO activado (Sin conexion a DB).")

    def get_employees(self):
        if self.is_mock:
            df = pd.DataFrame({
                'ID_EMPLEADO': range(1, 11),
                'TX_NOMBRE': ["Susana", "Carlos", "Ana", "Luis", "Marta", "Jorge", "Elena", "Pablo", "Sofia", "Miguel"],
                'F_ALTA': [datetime(2018 + i%5, (i%11)+1, 1) for i in range(10)],
                'F_NACIMIENTO': [datetime(1980 + i, 1, 1) for i in range(10)],
                'F_BAJA': [None, datetime(2023, 1, 1), None, None, None, None, None, None, None, None],
                'CX_EDOCIVIL': ['S', 'C', 'S', 'C', 'S', 'S', 'C', 'S', 'C', 'S']
            })
        else:
            df = pd.read_sql("SELECT * FROM EM_EMPLEADOS", self.engine)
        
        df['F_ALTA'] = pd.to_datetime(df['F_ALTA'])
        df['F_NACIMIENTO'] = pd.to_datetime(df['F_NACIMIENTO'])
        df['F_BAJA'] = pd.to_datetime(df['F_BAJA'])
        df['ANTIGUEDAD_ANOS'] = (datetime.now() - df['F_ALTA']).dt.days / 365.25
        df['EDAD'] = (datetime.now() - df['F_NACIMIENTO']).dt.days / 365.25
        df['ESTADO'] = df['F_BAJA'].apply(lambda x: 'Inactivo' if pd.notnull(x) else 'Activo')
        return df

    def get_projects(self):
        if self.is_mock:
            df = pd.DataFrame({
                'ID_PROYECTO': range(1, 6),
                'TX_DESCRIPCION': ["Web Corporativa", "App Movil", "Migracion Cloud", "Auditoria 2026", "IA Predictiva"],
                'F_INICIO': [datetime(2023, 1, 1) + timedelta(days=i*60) for i in range(5)],
                'F_FIN': [datetime(2024, 1, 1) + timedelta(days=i*60) for i in range(5)],
                'F_BAJA': [None, None, None, datetime(2023, 12, 1), None],
                'TX_LUGAR': ["Madrid", "Barcelona", "Valencia", "Madrid", "Sevilla"]
            })
        else:
            df = pd.read_sql("SELECT * FROM PR_PROYECTOS", self.engine)
        
        df['F_INICIO'] = pd.to_datetime(df['F_INICIO'])
        df['F_FIN'] = pd.to_datetime(df['F_FIN'])
        df['F_BAJA'] = pd.to_datetime(df['F_BAJA'])
        df['DURACION_DIAS'] = (df['F_FIN'].fillna(datetime.now()) - df['F_INICIO']).dt.days
        df['ESTADO'] = df['F_BAJA'].apply(lambda x: 'Cesado' if pd.notnull(x) else 'Activo')
        return df

    def get_assignments(self):
        if self.is_mock:
            return pd.DataFrame({
                'PROYECTO': ["Web Corporativa", "Web Corporativa", "App Movil", "App Movil", "IA Predictiva"],
                'ID_EMPLEADO': [1, 2, 3, 4, 1]
            })
        else:
            return pd.read_sql("SELECT p.TX_DESCRIPCION as PROYECTO, e.ID_EMPLEADO FROM PR_EMPLEADOS_PROYECTO ep JOIN PR_PROYECTOS p ON ep.ID_PROYECTO = p.ID_PROYECTO JOIN EM_EMPLEADOS e ON ep.ID_EMPLEADO = e.ID_EMPLEADO", self.engine)

# Iniciar motor
engine = DataEngine()
df_emp = engine.get_employees()
df_prj = engine.get_projects()
df_asg = engine.get_assignments()
print("Datos cargados correctamente.")

## 2. Capital Humano

In [17]:
# Antiguedad Top 5 + 5
top_old = df_emp.nlargest(5, 'ANTIGUEDAD_ANOS').copy()
top_new = df_emp.nsmallest(5, 'ANTIGUEDAD_ANOS').copy()
top_old['Cat'] = 'Veteranos'
top_new['Cat'] = 'Nuevos'
fig1 = px.bar(pd.concat([top_old, top_new]), x='TX_NOMBRE', y='ANTIGUEDAD_ANOS', color='Cat', title="Antiguedad del Personal")
fig1.show()

# Estado Civil (Corregido)
civil = df_emp.groupby('CX_EDOCIVIL').size().reset_index()
civil.columns = ['Code', 'Total']
civil['Label'] = civil['Code'].map({'S': 'Soltero/a', 'C': 'Casado/a', 'V': 'Viudo/a', 'D': 'Divorciado/a'})
fig2 = px.pie(civil, values='Total', names='Label', title="Estado Civil")
fig2.show()

## 3. Evolucion y Proyectos

In [18]:
# Evolucion
h = df_emp['F_ALTA'].dt.year.value_counts().reset_index()
h.columns = ['Y', 'Altas']
e = df_emp['F_BAJA'].dt.year.value_counts().reset_index()
e.columns = ['Y', 'Bajas']
ev = pd.merge(h, e, on='Y', how='outer').fillna(0).sort_values('Y')
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=ev['Y'], y=ev['Altas'], name='Altas', line=dict(color='green')))
fig3.add_trace(go.Scatter(x=ev['Y'], y=ev['Bajas'], name='Bajas', line=dict(color='red')))
fig3.show()

# Proyectos por Sede
loc = df_prj.groupby('TX_LUGAR').size().reset_index()
loc.columns = ['Sede', 'Total']
fig4 = px.bar(loc, x='Sede', y='Total', title="Proyectos por Sede", color='Total')
fig4.show()